# Gold-standard (Cell Ranger) — expression consistency after mapping (marketing appendix)

This notebook is an *optional* deep dive that turns the gold-standard pipeline into a stronger marketing artifact.

## Rationale

The Fig 4c notebook focuses on **feature-set consistency** (gene identity) across Ensembl releases under a fixed snapshot boundary.
This companion notebook asks a complementary question:

> After mapping each release-specific feature space into a single target release, do the **pseudo-bulk expression profiles** become consistent across releases?

This is not an accuracy benchmark; it is a sanity/marketing visualization that the mapped feature space behaves coherently.

## Inputs and prerequisites

- `.h5ad` files produced by `experiment_cellranger_idtrack/create_data.ipynb`
- Conversion caches produced by `experiment_cellranger_idtrack/analysis_gold_standard_fig4c.ipynb`

## Outputs

- `_outputs/_publication/figures/fig_gold_standard_pseudobulk_correlation.pdf`
- `_outputs/_publication/figures/fig_gold_standard_pseudobulk_consistency_suite.pdf` (optional multi-panel marketing figure)
- `_outputs/_publication/figures/fig_gold_standard_expression_consistency_gain.pdf`
- `_outputs/_publication/tables/gold_standard_pseudobulk_correlation_by_distance.csv`
- Cached per-release pseudo-bulk vectors under `idtrack/docs/_notebooks/idtrack_cache/experiments/gold_standard_cellranger/`


Optional manuscript table:
- `_outputs/_publication/tables/gold_standard_pseudobulk_distance_summary.tex`


In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    atomic_write_text,
    notebook_context,
    read_pickle,
    save_figure,
    write_pickle,
)

try:
    import anndata as ad
except Exception as e:  # noqa: S110
    raise ImportError('This notebook requires anndata/scanpy environment.') from e

ctx = notebook_context('gold_standard_cellranger', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures
MANUSCRIPT_TABLES = ctx.manuscript_tables

# Make sure nested imports see the same location.
os.environ.setdefault('IDTRACK_LOCAL_REPO', str(IDTRACK_LOCAL_REPO))

ANNDATA_DIR_RAW = os.environ.get('GOLD_STANDARD_ANNDATA_DIR', '').strip()
ANNDATA_DIR = Path(ANNDATA_DIR_RAW).expanduser().resolve() if ANNDATA_DIR_RAW else None

print('ANNDATA_DIR:', ANNDATA_DIR)
print('CACHE_DIR:', CACHE_DIR)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)
print('MANUSCRIPT_TABLES:', MANUSCRIPT_TABLES)



In [ ]:
# -------------------- Configuration --------------------

TARGET_RELEASE = 114
FINAL_DATABASE = None  # keep Ensembl backbone
STRATEGY = 'best'      # use 1→1 for stable pseudo-bulk alignment

# How to reduce each AnnData into a vector for correlation.
# - 'sum' is pseudo-bulk (recommended)
PSEUDOBULK_MODE = 'sum'

print('TARGET_RELEASE:', TARGET_RELEASE)
print('PSEUDOBULK_MODE:', PSEUDOBULK_MODE)


In [ ]:
# -------------------- Discover .h5ad files and pick dataset/assembly --------------------

if not ANNDATA_DIR or not ANNDATA_DIR.exists():
    raise FileNotFoundError('Set GOLD_STANDARD_ANNDATA_DIR to the folder containing generated .h5ad files.')

h5ads = sorted(ANNDATA_DIR.glob('*.h5ad'))
pat = re.compile(r'^(?P<dataset>.+?)_(?P<assembly>[^_]+)_(?P<release>\d+)\.h5ad$')

rows = []
for p in h5ads:
    m = pat.match(p.name)
    if not m:
        continue
    rows.append({'dataset': m.group('dataset'), 'assembly': m.group('assembly'), 'release': int(m.group('release')), 'path': str(p)})

files = pd.DataFrame(rows).sort_values(['dataset', 'assembly', 'release']).reset_index(drop=True)
if files.empty:
    raise RuntimeError('No .h5ad files matched the expected pattern: <dataset>_<assembly>_<release>.h5ad')

grp = files.groupby(['dataset', 'assembly']).size().reset_index(name='n_files')
best = grp.sort_values('n_files', ascending=False).iloc[0]
dataset = str(best['dataset'])
assembly = str(best['assembly'])

subset = files[(files['dataset'] == dataset) & (files['assembly'] == assembly)].sort_values('release').reset_index(drop=True)
print('Selected dataset:', dataset)
print('Selected assembly:', assembly)
print('Releases:', subset['release'].tolist())

subset.head()


In [ ]:
# -------------------- Load conversion caches and compute pseudo-bulk vectors --------------------

def _safe_stem(s: str) -> str:
    return ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(s))


def _conv_pickle_path(dataset: str, assembly: str, release: int) -> Path:
    return CACHE_DIR / (
        f"idtrack_matchings_{_safe_stem(dataset)}_{_safe_stem(assembly)}_from{release}_to{TARGET_RELEASE}"
        f"_final{FINAL_DATABASE or 'ensembl'}_strategy{STRATEGY}.pickle"
    )


def load_matchings(dataset: str, assembly: str, release: int) -> list[dict]:
    p = _conv_pickle_path(dataset, assembly, release)
    if not p.exists():
        raise FileNotFoundError(
            f"Missing conversion cache for release {release}: {p}. "
            "Run `analysis_gold_standard_fig4c.ipynb` to generate caches."
        )
    obj = read_pickle(p)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict) and 'matchings' in obj:
        return obj['matchings']
    raise TypeError(f'Unexpected conversion cache format in {p}: {type(obj)}')


def mapping_1_to_1(matchings: list[dict]) -> dict[str, str]:
    out: dict[str, str] = {}
    for m in matchings:
        if m.get('no_corresponding') or m.get('no_conversion') or m.get('no_target'):
            continue
        t = m.get('target_id') or []
        if len(t) != 1:
            continue
        q = str(m.get('query_id'))
        out[q] = str(t[0])
    return out


def pseudobulk_raw_vector(adata) -> pd.Series:
    X = adata.X
    sums = X.sum(axis=0)
    if hasattr(sums, 'A1'):
        sums = sums.A1
    sums = np.asarray(sums).ravel()

    src_ids = [str(x) for x in adata.var_names]
    vec = pd.Series(sums, index=src_ids, name='sum')
    vec = vec.groupby(level=0).sum()  # defensive: collapse duplicates
    return vec


def pseudobulk_target_vector(adata, q_to_t: dict[str, str]) -> pd.Series:
    # Build per-gene sums in the source space
    if PSEUDOBULK_MODE != 'sum':
        raise ValueError(f'Unsupported PSEUDOBULK_MODE: {PSEUDOBULK_MODE}')

    X = adata.X
    sums = X.sum(axis=0)
    if hasattr(sums, 'A1'):
        sums = sums.A1
    sums = np.asarray(sums).ravel()

    src_ids = [str(x) for x in adata.var_names]
    df = pd.DataFrame({'query_id': src_ids, 'sum': sums})
    df['target_id'] = df['query_id'].map(q_to_t)
    df = df.dropna(subset=['target_id'])

    # Collapse potential n→1 collisions by summing into target space
    vec = df.groupby('target_id')['sum'].sum()
    vec.name = 'sum'
    return vec


mapped_vectors: dict[int, pd.Series] = {}
raw_vectors: dict[int, pd.Series] = {}

for r in subset.itertuples(index=False):
    rel = int(r.release)

    out_mapped = CACHE_DIR / f"pseudobulk_{_safe_stem(dataset)}_{_safe_stem(assembly)}_{rel}_to{TARGET_RELEASE}.pickle"
    out_raw = CACHE_DIR / f"pseudobulk_raw_{_safe_stem(dataset)}_{_safe_stem(assembly)}_{rel}.pickle"

    have_mapped = out_mapped.exists()
    have_raw = out_raw.exists()

    if have_mapped:
        mapped_vectors[rel] = read_pickle(out_mapped)
    if have_raw:
        raw_vectors[rel] = read_pickle(out_raw)

    if have_mapped and have_raw:
        continue

    adata = ad.read_h5ad(str(r.path))

    if not have_raw:
        vec_raw = pseudobulk_raw_vector(adata)
        write_pickle(vec_raw, out_raw)
        raw_vectors[rel] = vec_raw

    if not have_mapped:
        matchings = load_matchings(dataset, assembly, rel)
        q_to_t = mapping_1_to_1(matchings)
        vec = pseudobulk_target_vector(adata, q_to_t)
        write_pickle(vec, out_mapped)
        mapped_vectors[rel] = vec

print('Pseudo-bulk vectors (mapped):', len(mapped_vectors))
print('Pseudo-bulk vectors (raw):', len(raw_vectors))
list(mapped_vectors)[:10]



In [ ]:
# -------------------- Build correlation matrices and plot --------------------

def correlation_matrix(vectors: dict[int, pd.Series], *, space_name: str) -> tuple[pd.DataFrame, list[int], list[str]]:
    releases = sorted(vectors)
    if len(releases) < 2:
        raise RuntimeError(f'Need at least 2 releases to compute a correlation matrix ({space_name}).')

    # Align on the intersection of genes across releases.
    common = set(vectors[releases[0]].index)
    for r in releases[1:]:
        common &= set(vectors[r].index)

    common = sorted(common)
    mat = []
    for r in releases:
        v = vectors[r].reindex(common).fillna(0.0).astype(float)
        mat.append(np.log1p(v.values))

    X = np.vstack(mat)
    corr = np.corrcoef(X)
    corr_df = pd.DataFrame(corr, index=releases, columns=releases)
    return corr_df, releases, common


corr_mapped, releases, common_mapped = correlation_matrix(mapped_vectors, space_name='mapped')
corr_raw, _, common_raw = correlation_matrix(raw_vectors, space_name='raw')

print('Common genes (mapped):', len(common_mapped))
print('Common genes (raw):', len(common_raw))

# Export matrices for manuscript / debugging
out_corr_mapped = CACHE_DIR / f"pseudobulk_correlation_mapped_{_safe_stem(dataset)}_{_safe_stem(assembly)}_to{TARGET_RELEASE}.csv"
out_corr_raw = CACHE_DIR / f"pseudobulk_correlation_raw_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"
atomic_write_text(out_corr_mapped, corr_mapped.to_csv())
atomic_write_text(out_corr_raw, corr_raw.to_csv())
print('Wrote:', out_corr_mapped)
print('Wrote:', out_corr_raw)

# Keep the original (single-panel) mapped heatmap for backwards compatibility
fig, ax = plt.subplots(1, 1, figsize=(6.2, 5.2))
if sns is not None:
    sns.heatmap(corr_mapped, ax=ax, cmap='Blues', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Pearson r'})
else:
    im = ax.imshow(corr_mapped.values, cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, label='Pearson r')
    ax.set_xticks(range(len(releases)))
    ax.set_yticks(range(len(releases)))
    ax.set_xticklabels(releases, rotation=30, ha='right')
    ax.set_yticklabels(releases)

ax.set_title('Gold standard: pseudo-bulk consistency after 1→1 mapping')
ax.set_xlabel('Starting release')
ax.set_ylabel('Starting release')
fig.tight_layout()

written = save_figure(fig, 'fig_gold_standard_pseudobulk_correlation.pdf', ctx, formats=('pdf',))
print('Saved:', written['pdf'])

# ------------------------------------------------------------------
# Extended multi-panel suite (raw vs mapped + distance trends)
# ------------------------------------------------------------------

from itertools import combinations

pair_rows = []
for r1, r2 in combinations(releases, 2):
    pair_rows.append(
        {
            'r1': r1,
            'r2': r2,
            'distance': abs(int(r1) - int(r2)),
            'corr_raw': float(corr_raw.loc[r1, r2]),
            'corr_mapped': float(corr_mapped.loc[r1, r2]),
            'n_common_raw_pair': int(len(set(raw_vectors[r1].index) & set(raw_vectors[r2].index))),
            'n_common_mapped_pair': int(len(set(mapped_vectors[r1].index) & set(mapped_vectors[r2].index))),
        }
    )

pairs = pd.DataFrame(pair_rows)

out_pairs = CACHE_DIR / f"pseudobulk_pairwise_{_safe_stem(dataset)}_{_safe_stem(assembly)}_to{TARGET_RELEASE}.csv"
atomic_write_text(out_pairs, pairs.to_csv(index=False))
print('Wrote:', out_pairs)

summ = (
    pairs.groupby('distance')
    .agg(
        corr_raw_median=('corr_raw', 'median'),
        corr_mapped_median=('corr_mapped', 'median'),
        n_common_raw_median=('n_common_raw_pair', 'median'),
        n_common_mapped_median=('n_common_mapped_pair', 'median'),
        n_pairs=('distance', 'size'),
    )
    .reset_index()
)

out_summ = CACHE_DIR / f"pseudobulk_distance_summary_{_safe_stem(dataset)}_{_safe_stem(assembly)}_to{TARGET_RELEASE}.csv"
atomic_write_text(out_summ, summ.to_csv(index=False))
print('Wrote:', out_summ)

fig2, axes = plt.subplots(2, 2, figsize=(12.6, 8.0), constrained_layout=True)
ax0, ax1, ax2, ax3 = axes.ravel()

if sns is not None:
    sns.heatmap(corr_raw, ax=ax0, cmap='Blues', vmin=0, vmax=1, square=True, cbar=False)
    ax0.set_title(f'A) Raw pseudo-bulk (|common|={len(common_raw)})')

    sns.heatmap(corr_mapped, ax=ax1, cmap='Blues', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Pearson r'})
    ax1.set_title(f'B) After IDTrack → r{TARGET_RELEASE} (|common|={len(common_mapped)})')
else:
    ax0.imshow(corr_raw.values, cmap='Blues', vmin=0, vmax=1)
    ax0.set_title('A) Raw pseudo-bulk')
    ax1.imshow(corr_mapped.values, cmap='Blues', vmin=0, vmax=1)
    ax1.set_title(f'B) After IDTrack → r{TARGET_RELEASE}')

for ax in (ax0, ax1):
    ax.set_xlabel('Starting release')
    ax.set_ylabel('Starting release')

# C) correlation vs release distance
ax2.plot(
    summ['distance'],
    summ['corr_raw_median'],
    '-o',
    label='Raw',
    color=MANUSCRIPT_COLORS['neutral'],
    lw=1.4,
    ms=3,
)
ax2.plot(
    summ['distance'],
    summ['corr_mapped_median'],
    '-o',
    label='Mapped (1→1)',
    color=MANUSCRIPT_COLORS['1→1'],
    lw=1.4,
    ms=3,
)
ax2.set_xlabel('|Δ release|')
ax2.set_ylabel('Median Pearson r')
ax2.set_title('C) Expression consistency vs release distance')
ax2.set_ylim(0, 1)
ax2.legend(frameon=True)

# D) comparable gene-set size vs distance
ax3.plot(
    summ['distance'],
    summ['n_common_raw_median'],
    '-o',
    label='Raw',
    color=MANUSCRIPT_COLORS['neutral'],
    lw=1.4,
    ms=3,
)
ax3.plot(
    summ['distance'],
    summ['n_common_mapped_median'],
    '-o',
    label='Mapped (1→1)',
    color=MANUSCRIPT_COLORS['1→1'],
    lw=1.4,
    ms=3,
)
ax3.set_xlabel('|Δ release|')
ax3.set_ylabel('Median # common genes (pairwise)')
ax3.set_title('D) Comparable feature space vs distance')
ax3.legend(frameon=True)

written2 = save_figure(fig2, 'fig_gold_standard_pseudobulk_consistency_suite.pdf', ctx, formats=('pdf',))
print('Saved:', written2['pdf'])

# Also write a small manuscript table (distance summary)
caption = (
    'Gold-standard pseudo-bulk consistency as a function of release distance, comparing raw IDs vs IDTrack-mapped IDs.'
).replace('_', r'\_')

tex_rows = []
for r in summ.itertuples(index=False):
    tex_rows.append(
        ' & '.join(
            [
                str(int(r.distance)),
                str(int(r.n_pairs)),
                f"{float(r.corr_raw_median):.4f}",
                f"{float(r.corr_mapped_median):.4f}",
                str(int(r.n_common_raw_median)),
                str(int(r.n_common_mapped_median)),
            ]
        )
        + ' \\'
    )

tex_lines = [
    r'\begin{table}[t]',
    r'\centering',
    f'\caption{{{caption}}}',
    r'\label{tab:gold-standard-pseudobulk-distance}',
    r'\begin{tabular}{rrrrrr}',
    r'\toprule',
    r'$|\Delta r|$ & #pairs & median $r$ (raw) & median $r$ (mapped) & median #genes (raw) & median #genes (mapped) \\',
    r'\midrule',
    *tex_rows,
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table}',
]

out_tex = MANUSCRIPT_TABLES / 'gold_standard_pseudobulk_distance_summary.tex'
tex = '\n'.join(tex_lines) + '\n'
atomic_write_text(out_tex, tex)
atomic_write_text((ctx.experiment_outputs / 'tables' / out_tex.name), tex)
print('Wrote:', out_tex)
print('Wrote:', (ctx.experiment_outputs / 'tables' / out_tex.name))

summ



# Marketing extension: consistency gain vs release distance

A compact Results-friendly story is to show how expression-level similarity decays with release distance, and how much of that decay is recovered after harmonization.

This section:

- groups release pairs by $|\Delta\mathrm{release}|$
- compares mean/median correlations in **raw** vs **mapped** feature space
- exports a small figure + CSV suitable for manuscript selection


In [ ]:
from experiments_utils import atomic_write_dataframe_csv, label_panels  # noqa: E402

def _pairs_from_corr(corr: pd.DataFrame, *, label: str) -> pd.DataFrame:
    rels = [int(x) for x in corr.index]
    rows = []
    for i, r1 in enumerate(rels):
        for r2 in rels[i + 1:]:
            rows.append(
                {
                    'space': label,
                    'r1': int(r1),
                    'r2': int(r2),
                    'abs_delta': int(abs(int(r2) - int(r1))),
                    'corr': float(corr.loc[r1, r2]),
                }
            )
    return pd.DataFrame(rows)

pairs_raw = _pairs_from_corr(corr_raw, label='raw')
pairs_mapped = _pairs_from_corr(corr_mapped, label='mapped')
pairs_all = pd.concat([pairs_raw, pairs_mapped], ignore_index=True)

curve = (
    pairs_all.groupby(['space', 'abs_delta'], as_index=False)['corr']
    .agg(mean='mean', median='median', std='std', n='count')
)
curve['sem'] = curve['std'] / np.sqrt(curve['n'].replace(0, np.nan))

out_curve = MANUSCRIPT_TABLES / 'gold_standard_pseudobulk_correlation_by_distance.csv'
atomic_write_dataframe_csv(curve, out_curve, index=False)
atomic_write_dataframe_csv(curve, ctx.experiment_outputs / 'tables' / out_curve.name, index=False)
print('Wrote:', out_curve)

figG, axesG = plt.subplots(1, 2, figsize=(12.5, 4.2), constrained_layout=True)
ax0, ax1 = axesG

for space, color in [('raw', MANUSCRIPT_COLORS['1→0']), ('mapped', MANUSCRIPT_COLORS['1→1'])]:
    sub = curve[(curve['space'] == space) & (curve['abs_delta'] > 0)].sort_values('abs_delta')
    if sub.empty:
        continue
    ax0.errorbar(sub['abs_delta'], sub['mean'], yerr=sub['sem'], fmt='-o', ms=3, lw=1.4, label=space, color=color)
ax0.set_ylim(0, 1)
ax0.set_xlabel('|Δ release|')
ax0.set_ylabel('Mean correlation')
ax0.set_title('A) Mean pseudo-bulk correlation vs release distance')
ax0.legend(frameon=True)

# Delta curve (mapped - raw)
wide = curve.pivot(index='abs_delta', columns='space', values='mean').reset_index()
if {'raw', 'mapped'}.issubset(wide.columns):
    wide['delta'] = wide['mapped'] - wide['raw']
    ax1.plot(wide['abs_delta'], wide['delta'], '-o', ms=3, lw=1.4, color=MANUSCRIPT_COLORS['1→n'])
    ax1.axhline(0, color=MANUSCRIPT_COLORS['grid'], lw=1)
    ax1.set_xlabel('|Δ release|')
    ax1.set_ylabel('Δ mean correlation')
    ax1.set_title('B) Consistency gain after harmonization')
else:
    ax1.axis('off')

label_panels(axesG)
writtenG = save_figure(figG, 'fig_gold_standard_expression_consistency_gain.pdf', ctx, formats=('pdf',))
print('Saved:', writtenG['pdf'])
